In [ ]:
import gzip
import pandas as pd

In [ ]:
# читаем файл построчно
rows = []
with gzip.open("Архив Погоды.csv.gz", "rt", encoding="utf-8") as f:
    for i, line in enumerate(f):
        if i < 7:
            continue
        line = line.strip().replace('"', '')
        parts = line.split(";")
        rows.append(parts)

df_raw = pd.DataFrame(rows)
df_raw

,0,1,2,3,4,5,6,7,8,9,...,22,23,24,25,26,27,28,29,30,31
0,31.12.2019 21:00,2.9,734.2,745.6,,86,"Ветер, дующий с запада",1,,,...,0.8,Осадков нет,12,,,,,,None,None
1,31.12.2019 18:00,2.7,733.7,745.1,,86,"Ветер, дующий с запада",1,,,...,0.6,Осадков нет,12,,,,,,None,None
2,31.12.2019 15:00,2.3,733.7,745.1,,87,"Ветер, дующий с запада",2,,,...,0.3,,,,,,,,None,None
3,31.12.2019 12:00,2.1,734.2,745.6,,86,"Ветер, дующий с запада",1,,,...,0.0,,,,,,,,None,None
4,31.12.2019 09:00,1.7,735.0,746.4,,87,"Ветер, дующий с запада",1,,,...,-0.2,0.5,12,,,Ровный слой слежавшегося или мокрого снега пок...,1.0,,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8616,01.01.2017 12:00,2.5,740.8,752.3,-0.2,92,"Ветер, дующий с западо-юго-запада",2,,,...,1.3,,,,,,,,None,None
8617,01.01.2017 09:00,2.2,741.0,752.5,-0.4,92,"Ветер, дующий с западо-юго-запада",2,,,...,1.0,Осадков нет,12,,,,,,None,None
8618,01.01.2017 06:00,1.7,741.4,753.0,-1.0,93,"Ветер, дующий с западо-юго-запада",1,,,...,0.7,Осадков нет,12,,,,,,None,None
8619,01.01.2017 03:00,1.9,742.4,754.0,-1.1,91,"Ветер, дующий с западо-юго-запада",1,,,...,0.6,,,,,,,,None,None


In [ ]:
# нужные столбцы по позиции:
# 0 - дата, время
# 1 - температура
# 5 - влажность
# 10 - облачность
df_weather = df_raw.iloc[:, [0, 1, 5, 10]].copy()
df_weather.columns = ["datetime", "temp_c", "humidity", "cloudiness"]

df_weather["datetime"] = pd.to_datetime(df_weather["datetime"], format="%d.%m.%Y %H:%M")

df_weather["temp_c"] = pd.to_numeric(df_weather["temp_c"], errors="coerce")
df_weather["humidity"] = pd.to_numeric(df_weather["humidity"], errors="coerce")

cloudiness_map = {
    "Облаков нет.": 0,
    "10%  или менее, но не 0": 5,
    "20–30%.": 25,
    "40%.": 40,
    "50%.": 50,
    "60%.": 60,
    "70 – 80%.": 75,
    "90  или более, но не 100%": 95,
    "100%.": 100,
    "Небо не видно из-за тумана и/или других метеорологических явлений.": None,
    "": None,
}
df_weather["cloudiness"] = df_weather["cloudiness"].map(cloudiness_map)

df_weather = df_weather.sort_values("datetime").reset_index(drop=True)

In [ ]:
df_weather.isna().sum()

,0
datetime,0
temp_c,0
humidity,0
cloudiness,5


In [ ]:
df_weather.head()

,datetime,temp_c,humidity,cloudiness
0,2017-01-01 00:00:00,1.3,92,100.0
1,2017-01-01 03:00:00,1.9,91,100.0
2,2017-01-01 06:00:00,1.7,93,100.0
3,2017-01-01 09:00:00,2.2,92,100.0
4,2017-01-01 12:00:00,2.5,92,100.0


In [ ]:
df_weather["cloudiness"] = df_weather["cloudiness"].interpolate(method="linear")

# переиндексируем с 3 часов на 30 минут
full_index = pd.date_range(
    start=df_weather["datetime"].min(),
    end=df_weather["datetime"].max(),
    freq="30min"
)

df_weather = df_weather.set_index("datetime").reindex(full_index)
df_weather.index.name = "timestamp"
df_weather = df_weather.reset_index()

df_weather["temp_c"] = df_weather["temp_c"].interpolate(method="linear")
df_weather["humidity"] = df_weather["humidity"].interpolate(method="linear")
df_weather["cloudiness"] = df_weather["cloudiness"].interpolate(method="linear")

In [ ]:
df_weather.head()

,timestamp,temp_c,humidity,cloudiness
0,2017-01-01 00:00:00,1.3,92.000000,100.0
1,2017-01-01 00:30:00,1.4,91.833333,100.0
2,2017-01-01 01:00:00,1.5,91.666667,100.0
3,2017-01-01 01:30:00,1.6,91.500000,100.0
4,2017-01-01 02:00:00,1.7,91.333333,100.0


In [ ]:
df_final = pd.read_csv("df_final.csv", parse_dates=["timestamp"])

In [ ]:
df_merged = df_final.merge(df_weather, on="timestamp", how="left")

In [ ]:
df_merged.head()

,timestamp,house_1,house_2,house_3,house_4,house_5,house_6,house_7,house_8,temp_c,humidity,cloudiness
0,2017-11-11 00:30:00,58.17,22.40,16.62,38.96,26.560,60.30,84.49,32.90,2.716667,91.0,100.0
1,2017-11-11 01:00:00,49.74,19.84,14.28,33.84,22.585,55.80,74.94,29.15,2.833333,91.0,100.0
2,2017-11-11 01:30:00,44.76,17.84,12.78,32.94,20.960,49.74,66.70,24.95,2.950000,91.0,100.0
3,2017-11-11 02:00:00,38.22,17.32,12.06,30.30,18.800,49.56,60.67,24.60,3.066667,91.0,100.0
4,2017-11-11 02:30:00,35.40,15.28,11.40,27.98,18.165,47.64,57.23,23.20,3.183333,91.0,100.0


In [ ]:
df_merged[["temp_c", "humidity", "cloudiness"]].isna().sum()

,0
temp_c,0
humidity,0
cloudiness,0


In [ ]:
df_merged.to_csv("df_final+whether.csv", index=False)

In [ ]:
houses = [col for col in df_merged.columns if col.startswith("house_")]
weather_cols = ["temp_c", "humidity", "cloudiness"]

# Корреляция Пирсона
corr_results = []

for house in houses:
    for weather in weather_cols:
        corr = df_merged[house].corr(df_merged[weather])
        corr_results.append({
            "Дом": house,
            "Погодный фактор": weather,
            "Корреляция": round(corr, 3)
        })

df_corr = pd.DataFrame(corr_results)

df_corr_pivot = df_corr.pivot(index="Дом", columns="Погодный фактор", values="Корреляция")
print(df_corr_pivot.to_string())

Погодный фактор  cloudiness  humidity  temp_c
Дом                                          
house_1               0.128     0.044  -0.234
house_2               0.123     0.067  -0.264
house_3               0.127     0.072  -0.272
house_4               0.123     0.092  -0.290
house_5               0.162     0.091  -0.337
house_6               0.143     0.032  -0.231
house_7               0.115    -0.027  -0.155
house_8               0.154     0.039  -0.252


Температура (temp_c) — отрицательная корреляция у всех домов (-0.15 до -0.34). Чем холоднее — тем выше потребление. Это логично — освещение, электрообогреватели. Strongest у дома 5 (-0.337) и дома 4 (-0.290).
Облачность (cloudiness) — слабая положительная корреляция (0.11-0.16). Чем облачнее — тем больше потребление (меньше естественного освещения).
Влажность (humidity) — очень слабая корреляция, почти нулевая. Дом 7 вообще слабо отрицательный (-0.027).
В целом корреляции умеренные — это нормально для 30-минутных данных где суточный цикл доминирует над погодой.

In [ ]:
!pip install kaleido==0.2.1 -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.9/79.9 MB 8.1 MB/s eta 0:00:00


In [ ]:
import plotly.graph_objects as go
import numpy as np

fig = go.Figure(go.Heatmap(
    z=df_corr_pivot.values,
    x=df_corr_pivot.columns.tolist(),
    y=df_corr_pivot.index.tolist(),
    colorscale="RdBu",
    zmid=0,
    text=df_corr_pivot.values.round(3),
    texttemplate="%{text}",
    colorbar=dict(title="Корреляция", thickness=15)
))

fig.update_layout(
    title="Корреляция между погодными факторами и электрической нагрузкой",
    xaxis_title="Погодный фактор",
    yaxis_title="Дом",
    width=700,
    height=400,
    font=dict(size=11),
    margin=dict(l=50, r=20, t=40, b=40)
)

fig.show()
fig.write_image("13_weather_correlation.png", scale=2)

Но корреляция на сырых 30-минутных данных занижена — суточный цикл "перекрывает" влияние погоды. Рассчитаем корреляцию максимальных значения электрической нагрузки - это даст более честную картину

In [ ]:
# максимальная нагрузка за день
df_daily = df_merged.copy()
df_daily["date"] = df_daily["timestamp"].dt.date

# для нагрузки берём максимум, для погоды - среднее за день
df_daily_load = df_daily.groupby("date")[houses].max().reset_index()
df_daily_weather = df_daily.groupby("date")[weather_cols].mean().reset_index()

df_daily = df_daily_load.merge(df_daily_weather, on="date")
df_daily["mean_all"] = df_daily[houses].mean(axis=1)

df_daily.head()

,date,house_1,house_2,house_3,house_4,house_5,house_6,house_7,house_8,temp_c,humidity,cloudiness,mean_all
0,2017-11-11,99.06,46.40,26.70,67.16,43.955,127.98,154.37,54.20,4.512766,90.074468,100.000000,77.478125
1,2017-11-12,113.22,43.64,30.36,76.04,45.270,126.78,162.70,58.05,4.431250,84.062500,85.989583,82.007500
2,2017-11-13,107.67,37.44,29.73,72.38,48.705,133.98,161.84,55.00,1.923958,84.010417,73.385417,80.843125
3,2017-11-14,108.06,41.60,27.42,70.52,41.955,126.18,159.20,57.50,1.990625,89.375000,99.739583,79.054375
4,2017-11-15,100.92,40.76,28.86,70.00,46.035,126.84,161.10,59.80,2.172917,81.125000,99.635417,79.289375


In [ ]:
corr_daily_max = []

for house in houses:
    for weather in weather_cols:
        corr = df_daily[house].corr(df_daily[weather])
        corr_daily_max.append({
            "дом": house,
            "погодный фактор": weather,
            "корреляция": round(corr, 3)
        })

df_corr_max = pd.DataFrame(corr_daily_max)
df_corr_max_pivot = df_corr_max.pivot(
    index="дом", columns="погодный фактор", values="корреляция"
)

print(df_corr_max_pivot.to_string())

погодный фактор  cloudiness  humidity  temp_c
дом                                          
house_1               0.282     0.447  -0.780
house_2               0.274     0.411  -0.734
house_3               0.249     0.429  -0.734
house_4               0.284     0.494  -0.787
house_5               0.336     0.561  -0.858
house_6               0.302     0.533  -0.806
house_7               0.241     0.444  -0.723
house_8               0.295     0.489  -0.788


In [ ]:
fig = go.Figure(go.Heatmap(
    z=df_corr_max_pivot.values,
    x=df_corr_max_pivot.columns.tolist(),
    y=df_corr_max_pivot.index.tolist(),
    colorscale="RdBu",
    zmid=0,
    zmin=-1,
    zmax=1,
    text=df_corr_max_pivot.values.round(3),
    texttemplate="%{text}",
    colorbar=dict(title="корреляция", thickness=15)
))

fig.update_layout(
    title="Корреляция погодных факторов и максимальной электрической нагрузки",
    xaxis_title="Погодный фактор",
    yaxis_title="Дом",
    width=700,
    height=400,
    font=dict(size=11),
    margin=dict(l=50, r=20, t=40, b=40),
    showlegend=False
)

fig.show()
fig.write_image("14_weather_correlation_daily_max.png", scale=2)

In [ ]:
z = np.polyfit(df_daily["temp_c"], df_daily["mean_all"], 2)
p = np.poly1d(z)
x_line = np.linspace(df_daily["temp_c"].min(), df_daily["temp_c"].max(), 100)

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=df_daily["temp_c"],
    y=df_daily["mean_all"],
    mode="markers",
    marker=dict(color=df_daily["temp_c"], colorscale="RdBu_r", size=5, opacity=0.7, colorbar=dict(title="°C", thickness=15)),
    name="дневной максимум"
))
fig.add_trace(go.Scatter(
    x=x_line, y=p(x_line),
    mode="lines", line=dict(color="black", width=2), name="тренд"
))
fig.update_layout(
    title="Зависимость максимальной нагрузки от температуры",
    xaxis_title="Температура, °C", yaxis_title="Мощность, кВт",
    width=700, height=400, font=dict(size=11),
    margin=dict(l=50, r=20, t=40, b=40),
    showlegend=False
)
fig.show()
fig.write_image("15_temp_vs_load.png", scale=2)

In [ ]:
z2 = np.polyfit(df_daily["humidity"], df_daily["mean_all"], 1)
p2 = np.poly1d(z2)
x_line2 = np.linspace(df_daily["humidity"].min(), df_daily["humidity"].max(), 100)

fig2 = go.Figure()
fig2.add_trace(go.Scatter(
    x=df_daily["humidity"],
    y=df_daily["mean_all"],
    mode="markers",
    marker=dict(color=df_daily["humidity"], colorscale="Blues", size=5, opacity=0.7, colorbar=dict(title="%", thickness=15)),
    name="дневной максимум"
))
fig2.add_trace(go.Scatter(
    x=x_line2, y=p2(x_line2),
    mode="lines", line=dict(color="black", width=2), name="тренд"
))
fig2.update_layout(
    title="Зависимость максимальной нагрузки от влажности",
    xaxis_title="Влажность, %", yaxis_title="Мощность, кВт",
    width=700, height=400, font=dict(size=11),
    margin=dict(l=50, r=20, t=40, b=40),
    showlegend=False
)
fig2.show()
fig2.write_image("16_humidity_vs_load.png", scale=2)

In [ ]:
z3 = np.polyfit(df_daily["cloudiness"], df_daily["mean_all"], 1)
p3 = np.poly1d(z3)
x_line3 = np.linspace(df_daily["cloudiness"].min(), df_daily["cloudiness"].max(), 100)

fig3 = go.Figure()
fig3.add_trace(go.Scatter(
    x=df_daily["cloudiness"],
    y=df_daily["mean_all"],
    mode="markers",
    marker=dict(color=df_daily["cloudiness"], colorscale="Greys", size=5, opacity=0.7, colorbar=dict(title="%", thickness=15)),
    name="дневной максимум"
))
fig3.add_trace(go.Scatter(
    x=x_line3, y=p3(x_line3),
    mode="lines", line=dict(color="black", width=2), name="тренд"
))
fig3.update_layout(
    title="Зависимость максимальной нагрузки от облачности",
    xaxis_title="Облачность, %", yaxis_title="Мощность, кВт",
    width=700, height=400, font=dict(size=11),
    margin=dict(l=50, r=20, t=40, b=40),
    showlegend=False
)
fig3.show()
fig3.write_image("17_cloudiness_vs_load.png", scale=2)

Температура наружного воздуха является наиболее значимым погодным фактором, влияющим на электрическую нагрузку жилых домов. Коэффициент корреляции Пирсона между дневным максимумом нагрузки и средней дневной температурой составляет от -0.72 (дом 7) до -0.86 (дом 5). Отрицательный знак корреляции означает что снижение температуры наружного воздуха влечёт за собой рост электропотребления — преимущественно за счёт увеличения времени работы осветительных приборов и электрообогревательного оборудования. График зависимости демонстрирует нелинейный (U-образный) характер — при температурах выше +20°C наблюдается незначительный рост нагрузки, что объясняется использованием кондиционеров в летний период.
Влажность воздуха демонстрирует умеренную положительную корреляцию с дневным максимумом нагрузки — от 0.41 до 0.56. Высокая влажность характерна для холодных и пасмурных дней, поэтому данная связь носит косвенный характер и во многом объясняется совместным влиянием температуры и влажности. Тем не менее включение влажности как признака в модели прогнозирования может повысить их точность.
Облачность показывает наиболее слабую корреляцию — от 0.24 до 0.34. Большой разброс точек на графике свидетельствует о том что облачность сама по себе слабо предсказывает уровень нагрузки. Это объясняется тем что значение 100% встречается в датасете наиболее часто и охватывает широкий диапазон температурных условий.

Анализ показал что из трёх рассмотренных погодных факторов температура наружного воздуха оказывает наибольшее влияние на электрическую нагрузку жилых домов — коэффициент корреляции превышает 0.72 по всем домам. Данный результат согласуется с выводами работ [1, 3] из обзора литературы и подтверждает целесообразность включения температуры как экзогенной переменной в модели прогнозирования. Влажность рекомендуется включить как дополнительный признак, тогда как облачность имеет ограниченную предсказательную силу и может быть исключена из финального набора признаков без существенной потери качества.